In [1]:
import tensorflow as tf
import numpy as np
import gc

from tensorflow.keras.datasets import cifar10
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam,RMSprop
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from skimage.feature import hog
from tensorflow.keras import backend as K
from sklearn.decomposition import PCA
from tensorflow.keras.regularizers import l2

In [2]:
# -----------------------------------
# 1. Define class names
# -----------------------------------

class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

# ============================================================
# 2. LOAD CIFAR-10 DATA
# ============================================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Original train shape:", x_train.shape)
print("Original test shape :", x_test.shape)


# ============================================================
# 3. NORMALIZE IMAGES
# ============================================================

x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0


# ============================================================
# 4. EXTRACT RGB HOG FEATURES
# ============================================================

def extract_hog_features(images):

    features = []

    for image in images:

        hog_feature = hog(
            image,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(3, 3),
            channel_axis=-1
        )

        features.append(hog_feature)

    return np.array(features, dtype=np.float32)


print("\nExtracting HOG features...")

x_train_hog = extract_hog_features(x_train)
x_test_hog = extract_hog_features(x_test)

print("HOG train shape:", x_train_hog.shape)
print("HOG test shape :", x_test_hog.shape)


# ============================================================
# 5. EXTRACT 4×4 SPATIAL COLOR FEATURES
# ============================================================

def extract_spatial_color_features(images):

    features = []

    for image in images:

        image_features = []

        # 4 × 4 grid
        for row in range(4):

            for col in range(4):

                region = image[
                    row * 8:(row + 1) * 8,
                    col * 8:(col + 1) * 8
                ]

                # R, G, B
                for channel in range(3):

                    channel_data = region[:, :, channel]

                    image_features.append(
                        np.mean(channel_data)
                    )

                    image_features.append(
                        np.std(channel_data)
                    )

                    image_features.append(
                        np.min(channel_data)
                    )

                    image_features.append(
                        np.max(channel_data)
                    )

        features.append(image_features)

    return np.array(features, dtype=np.float32)


print("\nExtracting spatial color features...")

x_train_color = extract_spatial_color_features(x_train)
x_test_color = extract_spatial_color_features(x_test)

print("Color train shape:", x_train_color.shape)
print("Color test shape :", x_test_color.shape)


# ============================================================
# 6. COMBINE HOG + COLOR FEATURES
# ============================================================

x_train_combined = np.concatenate(
    [x_train_hog, x_train_color],
    axis=1
)

x_test_combined = np.concatenate(
    [x_test_hog, x_test_color],
    axis=1
)

print("\nCombined feature shape:")
print("Train:", x_train_combined.shape)
print("Test :", x_test_combined.shape)


# ============================================================
# 7. DELETE UNNECESSARY FEATURE ARRAYS
# ============================================================

# We don't need these separate arrays anymore.
del x_train_hog
del x_test_hog
del x_train_color
del x_test_color

gc.collect()


# ============================================================
# 8. PCA - 85% VARIANCE
# ============================================================

print("\nStarting PCA...")

pca = PCA(
    n_components=0.90,
    random_state=42
)

x_train_pca = pca.fit_transform(x_train_combined)

x_test_pca = pca.transform(x_test_combined)


# ============================================================
# 9. PCA INFORMATION
# ============================================================

print("\nPCA Results")

print("Original features:", x_train_combined.shape[1])
print("Train shape:", x_train_pca.shape)
print("Test shape:", x_test_pca.shape)
print("PCA components:", pca.n_components_)
print("Explained variance:",
      np.sum(pca.explained_variance_ratio_))


# ============================================================
# 10. DELETE COMBINED FEATURES
# ============================================================

del x_train_combined
del x_test_combined

gc.collect()


# ============================================================
# 11. CLEAR KERAS SESSION
# ============================================================

K.clear_session()


Original train shape: (50000, 32, 32, 3)
Original test shape : (10000, 32, 32, 3)

Extracting HOG features...
HOG train shape: (50000, 2916)
HOG test shape : (10000, 2916)

Extracting spatial color features...
Color train shape: (50000, 192)
Color test shape : (10000, 192)

Combined feature shape:
Train: (50000, 3108)
Test : (10000, 3108)

Starting PCA...

PCA Results
Original features: 3108
Train shape: (50000, 371)
Test shape: (10000, 371)
PCA components: 371
Explained variance: 0.9003807



In [3]:
# -----------------------------------
# 4. Create the model
# -----------------------------------

model = Sequential([
    Dense(
        1024,
        activation="relu",
        input_shape=(x_train_pca.shape[1],),
        kernel_regularizer=l2(0.003)
    ),
    Dropout(0.45),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

# -----------------------------------
# 5. Compile
# -----------------------------------

model.compile(
    #optimizer=Adam(learning_rate=0.001),
    optimizer='rmsprop',
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# -----------------------------------
# 6. Early stopping
# -----------------------------------

# early_stopping = EarlyStopping(
#     monitor="val_loss",
#     patience=7,
#     restore_best_weights=True
# )

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=7,
    mode="max",
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

C:\Users\Vijay\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
# -----------------------------------
# 7. Train
# -----------------------------------

print("EarlyStopping:")
print("monitor =", early_stopping.monitor)
print("patience =", early_stopping.patience)
print("mode =", early_stopping.mode)
print("restore_best_weights =", early_stopping.restore_best_weights)

print("\nReduceLR:")
print("monitor =", reduce_lr.monitor)
print("patience =", reduce_lr.patience)
print("factor =", reduce_lr.factor)
print("min_lr =", reduce_lr.min_lr)

history = model.fit(
    x_train_pca,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

# -----------------------------------
# 8. Evaluate the model
# -----------------------------------

test_loss, test_accuracy = model.evaluate(
    x_test_pca,
    y_test
)

print("Test Accuracy:", test_accuracy)

EarlyStopping:
monitor = val_accuracy
patience = 7
mode = max
restore_best_weights = True

ReduceLR:
monitor = val_loss
patience = 3
factor = 0.5
min_lr = 1e-05
Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.5728 - loss: 1.7092 - val_accuracy: 0.6292 - val_loss: 1.3368 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.6374 - loss: 1.3036 - val_accuracy: 0.6528 - val_loss: 1.2770 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6615 - loss: 1.2582 - val_accuracy: 0.6719 - val_loss: 1.2598 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6846 - loss: 1.2340 - val_accuracy: 0.6797 - val_loss: 1.2690 - learning_rate: 0.0010
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6959 - loss: 1.2247 - val_accuracy: 0.6869 - val_loss: 1.2743 - learning_rate: 0.0010
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7113 - 

In [5]:
# Predict using the PCA-transformed test images
predictions = model.predict(x_test_pca)

# Convert probabilities to class indices
predicted_classes = np.argmax(predictions, axis=1)

# Display actual vs predicted
for i in range(10):
    actual = class_names[y_test[i][0]]
    predicted = class_names[predicted_classes[i]]

    print(
        "Actual:", actual,
        "| Predicted:", predicted
    )

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Actual: cat | Predicted: dog
Actual: ship | Predicted: ship
Actual: ship | Predicted: automobile
Actual: airplane | Predicted: airplane
Actual: frog | Predicted: cat
Actual: frog | Predicted: frog
Actual: automobile | Predicted: automobile
Actual: frog | Predicted: frog
Actual: cat | Predicted: cat
Actual: automobile | Predicted: automobile


In [6]:
model.save("cifar10_model.keras")